# LeetCode 409 — Longest Palindrome bằng Python

**Link đề:** https://leetcode.com/problems/longest-palindrome/
**Độ khó:** Easy · **Chủ đề:** Hash Table, String, Greedy

```python
class Solution:
    def longestPalindrome(self, s: str) -> int:
        ...
```

## Mục tiêu học
1. Hiểu cấu trúc của một **palindrome** (chuỗi đối xứng): ký tự đi theo **cặp**, tối đa **một** ký tự lẻ ở giữa.
2. Đếm tần suất bằng `dict`, `dict.get`, `collections.Counter`, và theo dõi số lẻ bằng `set`.
3. Hiểu tư duy **tham lam** (greedy): lấy càng nhiều cặp càng tốt, thêm 1 ở giữa nếu còn ký tự lẻ.
4. Viết 4–5 lời giải: thử mọi cách chọn (brute force, chỉ để kiểm tra) → đếm bằng dict → Counter → set bật/tắt → sắp xếp ghép cặp.
5. Trace, chứng minh, Time/Space, lỗi hay gặp, biến thể (dựng luôn chuỗi palindrome).

## 1. Đề bài

Cho chuỗi `s` gồm chữ thường và/hoặc chữ hoa. Trả về **độ dài** của palindrome dài nhất có thể **xây dựng** từ các chữ cái đó (mỗi chữ cái dùng tối đa số lần nó xuất hiện, được sắp xếp lại tùy ý).

Phân biệt hoa/thường: `"Aa"` **không** phải palindrome.

| `s` | Output | Giải thích |
|---|:---:|---|
| `"abccccdd"` | `7` | ví dụ `"dccaccd"` |
| `"a"` | `1` | chính `"a"` |
| `"Aa"` | `1` | `A` và `a` khác nhau, chỉ lấy được 1 ký tự |

**Constraints:**
- `1 <= s.length <= 2000`
- `s` chỉ gồm chữ cái tiếng Anh hoa và/hoặc thường.

Lưu ý: đề chỉ hỏi **độ dài**, không cần in ra palindrome. Và ta được **sắp xếp lại** ký tự — khác hẳn bài "Longest Palindromic Substring" (LeetCode 5) phải giữ nguyên thứ tự.

## 2. Nền tảng: palindrome được cấu tạo thế nào?

Palindrome đọc xuôi = đọc ngược. Nhìn vị trí đối xứng:

```text
d c c a c c d
↑           ↑   cặp d
  ↑       ↑     cặp c
    ↑   ↑       cặp c
      ↑         'a' ở giữa (đứng một mình)
```

**Quy luật:** mỗi ký tự ở nửa trái phải có "bạn" ở nửa phải. Chỉ **vị trí chính giữa** (khi độ dài lẻ) là được đứng một mình.

⇒ Một ký tự xuất hiện `c` lần thì dùng được nhiều nhất `c` lần nếu `c` chẵn; nếu `c` lẻ thì dùng được `c - 1` lần **làm cặp**, và có thể đặt 1 ký tự lẻ ở giữa (chỉ một ký tự trong toàn bộ chuỗi được làm vậy).

In [1]:
p = "dccaccd"
print(p, "==", p[::-1], "->", p == p[::-1])
mid = len(p) // 2
print("ký tự giữa:", p[mid])
for i in range(mid):
    print(f"  cặp vị trí ({i}, {len(p) - 1 - i}):", p[i], p[len(p) - 1 - i])

dccaccd == dccaccd -> True
ký tự giữa: a
  cặp vị trí (0, 6): d d
  cặp vị trí (1, 5): c c
  cặp vị trí (2, 4): c c


### 2.1. Đếm tần suất bằng `dict`

`dict` (dictionary) lưu cặp **key → value**. Ở đây key là ký tự, value là số lần xuất hiện.

`counts.get(ch, 0)` trả về giá trị của `ch` nếu có, ngược lại trả `0` — tránh lỗi `KeyError` khi ký tự xuất hiện lần đầu.

In [2]:
counts = {}
for ch in "abccccdd":
    counts[ch] = counts.get(ch, 0) + 1
print(counts)
print("'c' xuất hiện", counts["c"], "lần")
print("'z' ->", counts.get("z", 0))

{'a': 1, 'b': 1, 'c': 4, 'd': 2}
'c' xuất hiện 4 lần
'z' -> 0


### 2.2. `collections.Counter` — dict chuyên đếm

`Counter(s)` đếm sẵn mọi phần tử. `.values()` cho các số đếm; `.most_common()` sắp theo số lần giảm dần.

In [3]:
from collections import Counter

cnt = Counter("abccccdd")
print(cnt)
print(list(cnt.values()))
print(cnt.most_common(2))
print(Counter("Aa"))   # phân biệt hoa thường

Counter({'c': 4, 'd': 2, 'a': 1, 'b': 1})
[1, 1, 4, 2]
[('c', 4), ('d', 2)]
Counter({'A': 1, 'a': 1})


### 2.3. Chẵn / lẻ với `%` và `//`

- `c % 2` bằng `0` nếu `c` chẵn, `1` nếu lẻ.
- `c // 2 * 2` là số chẵn lớn nhất không vượt `c` (= số ký tự dùng được làm cặp).
- Cũng viết được `c - c % 2` hoặc `c & ~1`.

In [4]:
for c in [1, 2, 3, 4, 5]:
    print(f"c={c}: c%2={c % 2}, c//2*2={c // 2 * 2}, c - c%2={c - c % 2}")

c=1: c%2=1, c//2*2=0, c - c%2=0
c=2: c%2=0, c//2*2=2, c - c%2=2
c=3: c%2=1, c//2*2=2, c - c%2=2
c=4: c%2=0, c//2*2=4, c - c%2=4
c=5: c%2=1, c//2*2=4, c - c%2=4


## 3. Cách trực giác (baseline) — thử mọi cách chọn số lượng

Không biết quy luật, ta có thể **thử hết**: với mỗi ký tự `ch` xuất hiện `c` lần, chọn dùng `0..c` lần. Một cách chọn **xếp được** thành palindrome khi có **tối đa một** ký tự được dùng số lẻ lần. Lấy tổng lớn nhất.

`itertools.product(*ranges)` sinh **mọi tổ hợp** chọn — số tổ hợp là tích `(c1+1)(c2+1)...` nên chỉ chạy nổi với chuỗi rất ngắn. Ta dùng làm **oracle**.

In [5]:
from itertools import product

print(list(product(range(2), range(3))))   # mọi cặp (a, b), a in 0..1, b in 0..2

[(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2)]


In [6]:
def longest_palindrome_bruteforce(s: str) -> int:
    counts = list(Counter(s).values())
    best = 0
    for choice in product(*(range(c + 1) for c in counts)):
        odd = sum(1 for k in choice if k % 2 == 1)
        if odd <= 1:
            best = max(best, sum(choice))
    return best

print(longest_palindrome_bruteforce("abccccdd"))  # 7
print(longest_palindrome_bruteforce("a"))         # 1
print(longest_palindrome_bruteforce("Aa"))        # 1

7
1
1


Cú pháp mới: `product(*(...))` — dấu `*` "rải" (unpack) các phần tử của một dãy thành nhiều đối số. `(range(c+1) for c in counts)` là **generator** tạo từng `range`.

## 4. Nút thắt và insight

**Nút thắt:** số tổ hợp tăng theo tích các số đếm — với `s` dài 2000 thì không thể thử hết.

**Insight (tham lam):** các ký tự **độc lập** với nhau, ta tối ưu từng ký tự riêng:
1. Với mỗi số đếm `c`, lấy phần chẵn `c // 2 * 2` — **luôn** có lợi, không bao giờ làm hỏng tính đối xứng.
2. Nếu **có ít nhất một** `c` lẻ, ta còn dư ít nhất một ký tự → đặt **đúng một** ký tự vào giữa: `+1`.

Công thức: `answer = Σ (c // 2 * 2) + (1 nếu tồn tại c lẻ)`.

Dạng tương đương: `answer = len(s) - số_ký_tự_lẻ + (1 nếu số_ký_tự_lẻ > 0)`.

In [7]:
cnt = Counter("abccccdd")
pairs_part = sum(c // 2 * 2 for c in cnt.values())
odd_count = sum(1 for c in cnt.values() if c % 2)
print("số đếm:", dict(cnt))
print("phần chẵn:", pairs_part, "| số ký tự có số đếm lẻ:", odd_count)
print("đáp án:", pairs_part + (1 if odd_count else 0))
print("dạng 2:", len("abccccdd") - odd_count + (1 if odd_count else 0))

số đếm: {'a': 1, 'b': 1, 'c': 4, 'd': 2}
phần chẵn: 6 | số ký tự có số đếm lẻ: 2
đáp án: 7
dạng 2: 7


## 5. Lời giải chính — đếm bằng `dict` + tham lam

**Đoạn 1 — đếm tần suất:**
```python
counts = {}
for ch in s:
    counts[ch] = counts.get(ch, 0) + 1
```
**Đoạn 2 — cộng phần chẵn, ghi nhận có lẻ hay không:**
```python
length = 0
has_odd = False
for c in counts.values():
    length += c // 2 * 2
    if c % 2 == 1:
        has_odd = True
```
**Đoạn 3 — thêm tâm:**
```python
if has_odd:
    length += 1
return length
```

In [8]:
def longest_palindrome_dict(s: str) -> int:
    counts = {}
    for ch in s:
        counts[ch] = counts.get(ch, 0) + 1
    length = 0
    has_odd = False
    for c in counts.values():
        length += c // 2 * 2
        if c % 2 == 1:
            has_odd = True
    if has_odd:
        length += 1
    return length

for s in ["abccccdd", "a", "Aa", "bb", "ccc"]:
    print(repr(s), "->", longest_palindrome_dict(s))

'abccccdd' -> 7
'a' -> 1
'Aa' -> 1
'bb' -> 2
'ccc' -> 3


## 6. TRACE từng bước

Bảng trace cho `s = "abccccdd"`, counts = `{a:1, b:1, c:4, d:2}`:

| Ký tự | c | c // 2 * 2 | c lẻ? | length sau | has_odd sau |
|:---:|:---:|:---:|:---:|:---:|:---:|
| a | 1 | 0 | có | 0 | True |
| b | 1 | 0 | có | 0 | True |
| c | 4 | 4 | không | 4 | True |
| d | 2 | 2 | không | 6 | True |

Kết thúc: `has_odd = True` → `6 + 1 = 7`. Lưu ý: dù có **hai** ký tự lẻ (`a`, `b`), ta chỉ cộng **1** — chỉ một ký tự đứng giữa được.

In [9]:
def trace_longest_palindrome(s: str) -> int:
    counts = {}
    for ch in s:
        counts[ch] = counts.get(ch, 0) + 1
    print("counts =", counts)
    length, has_odd = 0, False
    for ch, c in counts.items():
        add = c // 2 * 2
        length += add
        if c % 2 == 1:
            has_odd = True
        print(f"  '{ch}': c={c}, cộng {add} -> length={length}, has_odd={has_odd}")
    if has_odd:
        length += 1
        print("  có ký tự lẻ -> +1 ở giữa")
    print("  kết quả:", length)
    return length

trace_longest_palindrome("abccccdd")
print()
trace_longest_palindrome("aaabbbccc")

counts = {'a': 1, 'b': 1, 'c': 4, 'd': 2}
  'a': c=1, cộng 0 -> length=0, has_odd=True
  'b': c=1, cộng 0 -> length=0, has_odd=True
  'c': c=4, cộng 4 -> length=4, has_odd=True
  'd': c=2, cộng 2 -> length=6, has_odd=True
  có ký tự lẻ -> +1 ở giữa
  kết quả: 7

counts = {'a': 3, 'b': 3, 'c': 3}
  'a': c=3, cộng 2 -> length=2, has_odd=True
  'b': c=3, cộng 2 -> length=4, has_odd=True
  'c': c=3, cộng 2 -> length=6, has_odd=True
  có ký tự lẻ -> +1 ở giữa
  kết quả: 7


7

Với `"aaabbbccc"`: mỗi ký tự 3 lần → lấy 2+2+2 = 6, thêm 1 → 7 (ví dụ `"abcacba"`: dùng 3 chữ `a`, 2 chữ `b`, 2 chữ `c`). Không thể là 9 vì có **ba** ký tự lẻ mà chỉ một chỗ ở giữa.

`dict.items()` trả từng cặp `(key, value)` — tiện khi cần cả ký tự lẫn số đếm.

## 7. Các lời giải khác

Mọi cách dưới đây đều dùng **cùng một công thức**; chỉ khác cách đếm.

### 7.1. Cách 3 — `Counter` gọn gàng

In [10]:
def longest_palindrome_counter(s: str) -> int:
    counts = Counter(s).values()
    odd = sum(c % 2 for c in counts)          # số ký tự có số đếm lẻ
    return len(s) - odd + (1 if odd > 0 else 0)

for s in ["abccccdd", "a", "Aa", "aaabbbccc"]:
    print(repr(s), "->", longest_palindrome_counter(s))

'abccccdd' -> 7
'a' -> 1
'Aa' -> 1
'aaabbbccc' -> 7


### 7.2. Cách 4 — `set` bật/tắt (không cần đếm)

Duyệt từng ký tự: nếu `ch` đang ở trong set → vừa tạo được một **cặp** → xóa khỏi set, `pairs += 1`; nếu chưa có → thêm vào. Cuối cùng set chứa đúng các ký tự còn **lẻ**.

`set` là tập hợp không trùng lặp; `add`, `remove`, `in` trung bình O(1).

In [11]:
odd_set = set()
for ch in "abca":
    if ch in odd_set:
        odd_set.remove(ch)
        print(f"'{ch}' đã có -> ghép cặp, set = {odd_set}")
    else:
        odd_set.add(ch)
        print(f"'{ch}' chưa có -> thêm, set = {odd_set}")

'a' chưa có -> thêm, set = {'a'}
'b' chưa có -> thêm, set = {'a', 'b'}
'c' chưa có -> thêm, set = {'a', 'b', 'c'}
'a' đã có -> ghép cặp, set = {'b', 'c'}


In [12]:
def longest_palindrome_set(s: str) -> int:
    odd_set = set()
    pairs = 0
    for ch in s:
        if ch in odd_set:
            odd_set.remove(ch)
            pairs += 1
        else:
            odd_set.add(ch)
    return pairs * 2 + (1 if odd_set else 0)

for s in ["abccccdd", "a", "Aa", "aaabbbccc"]:
    print(repr(s), "->", longest_palindrome_set(s))

'abccccdd' -> 7
'a' -> 1
'Aa' -> 1
'aaabbbccc' -> 7


### 7.3. Cách 5 — sắp xếp rồi ghép cặp liền kề

Sau `sorted(s)`, các ký tự giống nhau đứng cạnh nhau. Duyệt với chỉ số `i`: nếu `t[i] == t[i+1]` → một cặp, nhảy 2; ngược lại ký tự lẻ, nhảy 1. Ý tưởng tốt khi không muốn dùng hash, nhưng tốn `O(n log n)`.

In [13]:
def longest_palindrome_sort(s: str) -> int:
    t = sorted(s)
    i, length, has_single = 0, 0, False
    while i < len(t):
        if i + 1 < len(t) and t[i] == t[i + 1]:
            length += 2
            i += 2
        else:
            has_single = True
            i += 1
    return length + (1 if has_single else 0)

print(sorted("abccccdd"))
for s in ["abccccdd", "a", "Aa", "aaabbbccc"]:
    print(repr(s), "->", longest_palindrome_sort(s))

['a', 'b', 'c', 'c', 'c', 'c', 'd', 'd']
'abccccdd' -> 7
'a' -> 1
'Aa' -> 1
'aaabbbccc' -> 7


## 8. Code nộp LeetCode

Bản chính dùng `dict` (không import); bản phụ dùng `set` với công thức `len(s) - len(odd) + (1 if odd else 0)`.

In [14]:
class Solution:
    def longestPalindrome(self, s: str) -> int:
        counts = {}
        for ch in s:
            counts[ch] = counts.get(ch, 0) + 1
        length = 0
        has_odd = False
        for c in counts.values():
            length += c // 2 * 2
            if c % 2 == 1:
                has_odd = True
        return length + (1 if has_odd else 0)


sol = Solution()
print(sol.longestPalindrome("abccccdd"))  # 7
print(sol.longestPalindrome("a"))         # 1

7
1


In [15]:
class SolutionSet:
    def longestPalindrome(self, s: str) -> int:
        odd = set()
        for ch in s:
            if ch in odd:
                odd.remove(ch)
            else:
                odd.add(ch)
        return len(s) - len(odd) + (1 if odd else 0)

print(SolutionSet().longestPalindrome("abccccdd"))

7


## 9. Vì sao đúng?

**Cận trên:** xét một palindrome bất kỳ dựng từ `s`. Trừ vị trí giữa (nếu độ dài lẻ), mọi vị trí đi theo cặp đối xứng chứa **cùng** ký tự. Vậy ký tự `ch` được dùng `u_ch` lần thì `u_ch` chẵn, trừ tối đa một ký tự. Mà `u_ch <= c_ch` nên `u_ch <= c_ch // 2 * 2` với mọi ký tự trừ một ký tự có thể thêm 1. ⇒ độ dài `<= Σ c//2*2 + 1`, và chỉ được `+1` khi có số đếm lẻ (nếu mọi `c` chẵn thì `c//2*2 = c`, không còn ký tự thừa).

**Đạt được:** đặt `c//2` bản của mỗi ký tự vào nửa trái, đảo lại làm nửa phải; nếu có ký tự lẻ, đặt một bản của nó ở giữa. Chuỗi này là palindrome có đúng độ dài công thức.

Cận trên = đạt được ⇒ công thức là tối ưu. Đây là kiểu chứng minh "**tham lam đúng**": lựa chọn cục bộ (lấy tối đa cặp) không làm mất cơ hội nào.

## 10. Độ phức tạp

Gọi `n = len(s)`, `k` = số ký tự khác nhau (`k <= 52`).

| Cách | Time | Space |
|---|:---:|:---:|
| Brute force thử mọi số lượng | `Π (c_i + 1)` — bùng nổ | `O(k)` |
| Dict / Counter | `O(n)` | `O(k)` = `O(1)` vì `k <= 52` |
| Set bật/tắt | `O(n)` | `O(k)` |
| Sắp xếp | `O(n log n)` | `O(n)` cho list đã sắp |

Mọi cách tốt đều phải đọc từng ký tự một lần ⇒ `O(n)` là tối ưu.

## 11. Bộ test

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name:<32} -> {actual}")

sol = Solution()
cases = [
    ("đề 1", "abccccdd", 7),
    ("đề 2", "a", 1),
    ("phân biệt hoa thường", "Aa", 1),
    ("toàn chẵn -> không +1", "aabb", 4),
    ("một ký tự lặp lẻ", "ccc", 3),
    ("nhiều ký tự lẻ chỉ +1", "aaabbbccc", 7),
    ("toàn khác nhau", "abcdef", 1),
    ("chuỗi dài 2000 cùng ký tự", "z" * 2000, 2000),
    ("hỗn hợp hoa thường", "AaBbAaBb", 8),
]
for name, s, exp in cases:
    check(name, sol.longestPalindrome(s), exp)

PASS  đề 1                             -> 7
PASS  đề 2                             -> 1
PASS  phân biệt hoa thường             -> 1
PASS  toàn chẵn -> không +1            -> 4
PASS  một ký tự lặp lẻ                 -> 3
PASS  nhiều ký tự lẻ chỉ +1            -> 7
PASS  toàn khác nhau                   -> 1
PASS  chuỗi dài 2000 cùng ký tự        -> 2000
PASS  hỗn hợp hoa thường               -> 8


In [17]:
import random
rng = random.Random(409)
funcs = [longest_palindrome_dict, longest_palindrome_counter, longest_palindrome_set,
         longest_palindrome_sort, Solution().longestPalindrome, SolutionSet().longestPalindrome]
for _ in range(400):
    s = "".join(rng.choice("aAbBc") for _ in range(rng.randint(1, 9)))
    exp = longest_palindrome_bruteforce(s)
    for f in funcs:
        assert f(s) == exp, (f.__name__, s)
for _ in range(200):     # chuỗi dài: chỉ đối chiếu các cách nhanh với nhau
    s = "".join(rng.choice("abcdefABCDEF") for _ in range(rng.randint(1, 2000)))
    assert len({f(s) for f in funcs}) == 1
print("400 chuỗi ngắn khớp brute force + 200 chuỗi dài 6 cách đồng thuận -> PASS")

400 chuỗi ngắn khớp brute force + 200 chuỗi dài 6 cách đồng thuận -> PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1: Cộng `+1` cho **mỗi** ký tự lẻ
```python
for c in counts.values():
    length += c if c % 2 == 0 else c   # cộng nguyên c lẻ
```
**Vì sao sai:** `"aaabbbccc"` → 9, nhưng chỉ một ký tự được đứng giữa. **Sửa:** cộng `c - 1` cho số lẻ, cuối cùng `+1` **một lần**.

### Lỗi 2: Bỏ hẳn ký tự có số đếm lẻ
```python
length += c if c % 2 == 0 else 0
```
`"ccc"` → chỉ còn `0 + 1 = 1` thay vì 3. Số lẻ vẫn cho `c - 1` ký tự làm cặp.

In [18]:
def wrong_add_all_odd(s):
    return sum(Counter(s).values())          # = len(s)

def wrong_drop_odd(s):
    length, has_odd = 0, False
    for c in Counter(s).values():
        if c % 2 == 0:
            length += c
        else:
            has_odd = True
    return length + (1 if has_odd else 0)

for s in ["aaabbbccc", "ccc"]:
    print(repr(s), "| lỗi 1:", wrong_add_all_odd(s), "| lỗi 2:", wrong_drop_odd(s), "| đúng:", Solution().longestPalindrome(s))

'aaabbbccc' | lỗi 1: 9 | lỗi 2: 1 | đúng: 7
'ccc' | lỗi 1: 3 | lỗi 2: 1 | đúng: 3


### Lỗi 3: Luôn `+1` dù không có ký tự lẻ
`"aabb"` → 5 sai (chỉ có 4 ký tự!). **Sửa:** chỉ `+1` khi `has_odd`.

### Lỗi 4: Đổi về chữ thường bằng `s.lower()`
Đề phân biệt hoa/thường: `"Aa"` → đáp án 1, nhưng `lower()` làm thành `"aa"` → 2 sai.

### Lỗi 5: Nhầm với bài "palindromic substring"
Bài này được **sắp xếp lại**; không cần giữ thứ tự, không cần quy hoạch động.

In [19]:
def wrong_always_plus_one(s):
    return sum(c // 2 * 2 for c in Counter(s).values()) + 1

print("lỗi 3 'aabb':", wrong_always_plus_one("aabb"), "(đúng 4)")
print("lỗi 4 'Aa':", Solution().longestPalindrome("Aa".lower()), "(đúng 1)")

lỗi 3 'aabb': 5 (đúng 4)
lỗi 4 'Aa': 2 (đúng 1)


## 13. Mẹo nhận dạng pattern

- "**Có thể sắp xếp lại / xây dựng** palindrome" → chỉ quan tâm **tần suất**: tối đa một ký tự lẻ.
- "Đếm số lần xuất hiện" → `dict.get(ch, 0) + 1` hoặc `Counter`.
- "Ghép cặp" → set bật/tắt hoặc `c // 2`.
- Tối ưu "từng phần độc lập" → tham lam từng phần rồi cộng lại.

Bài liên quan: 266 Palindrome Permutation (Premium), 2131 Longest Palindrome by Concatenating Two Letter Words, 1400 Construct K Palindrome Strings.

## 14. Biến thể

1. **Có thể hoán vị thành palindrome không?** (LeetCode 266) → số ký tự lẻ `<= 1`.
2. **Dựng luôn một palindrome dài nhất** (ví dụ nhỏ nhất theo thứ tự từ điển).
3. **Không phân biệt hoa/thường** → `s.lower()` trước khi đếm.

In [20]:
def can_permute_palindrome(s: str) -> bool:
    return sum(c % 2 for c in Counter(s).values()) <= 1

def build_longest_palindrome(s: str) -> str:
    cnt = Counter(s)
    left = []
    middle = ""
    for ch in sorted(cnt):
        left.append(ch * (cnt[ch] // 2))
        if cnt[ch] % 2 == 1 and middle == "":
            middle = ch
    half = "".join(left)
    return half + middle + half[::-1]

print(can_permute_palindrome("carerac"), can_permute_palindrome("code"))
p = build_longest_palindrome("abccccdd")
print(p, len(p), p == p[::-1])

True False
ccdadcc 7 True


## 15. Bài tập (TODO)

**Bài 1 (tính tay):** `s = "AAbbbccD"`. Lập bảng số đếm, phần chẵn, ký tự lẻ. Đáp án?

**Bài 2 (đoán output):** `longest_palindrome_set("abab")` — set rỗng hay không ở cuối, `pairs` bằng mấy?

**Bài 3 (điền chỗ trống):** hoàn thiện cách dict.

In [21]:
def exercise_longest_palindrome(s: str) -> int:
    counts = {}
    # TODO 1: đếm tần suất
    length, has_odd = 0, False
    # TODO 2: cộng phần chẵn, ghi nhận lẻ
    # TODO 3: +1 nếu có lẻ
    pass

# assert exercise_longest_palindrome("abccccdd") == 7
# assert exercise_longest_palindrome("aabb") == 4

**Bài 4 (tự viết):** `exercise_can_permute(s)` trả `True` nếu hoán vị được thành palindrome, dùng **set bật/tắt**.

**Bài 5 (biến thể):** `exercise_longest_ignore_case(s)` — như đề nhưng **không** phân biệt hoa/thường.

**Bài 6 (mini challenge):** `exercise_min_removals(s)` — số ký tự **ít nhất** phải xóa để phần còn lại hoán vị được thành palindrome.

In [22]:
def exercise_can_permute(s: str) -> bool:
    # TODO
    pass

def exercise_longest_ignore_case(s: str) -> int:
    # TODO
    pass

def exercise_min_removals(s: str) -> int:
    # TODO
    pass

# assert exercise_can_permute("aab") is True and exercise_can_permute("abc") is False
# assert exercise_longest_ignore_case("Aa") == 2
# assert exercise_min_removals("abc") == 2

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Đếm từng ký tự | `A:2, b:3, c:2, D:1` | Phần chẵn 2+2+2, có lẻ → +1 |
| 2 | Bật/tắt từng ký tự | `a` vào, `b` vào, `a` ra, `b` ra | `pairs = 2`, set rỗng |
| 3 | Theo mục 5 | `counts.get(ch, 0) + 1` | `length += c // 2 * 2` |
| 4 | Set chứa ký tự lẻ | Cuối cùng xem `len(set)` | `return len(odd) <= 1` |
| 5 | Chuẩn hóa trước | `s.lower()` | Gọi lại lời giải chính |
| 6 | Liên hệ với độ dài tối đa | Phải xóa bớt ký tự lẻ | `len(s) - longestPalindrome(s)` |

## 17. Đáp án tham khảo

- **Bài 1:** `A:2, b:3, c:2, D:1` → phần chẵn `2 + 2 + 2 + 0 = 6`, có lẻ (`b`, `D`) → **7**.
- **Bài 2:** `a` thêm, `b` thêm, `a` ghép (pairs=1), `b` ghép (pairs=2) → set rỗng → `2*2 + 0 = ` **4**.

In [23]:
print("Bài 1:", Solution().longestPalindrome("AAbbbccD"))
print("Bài 2:", longest_palindrome_set("abab"))

Bài 1: 7
Bài 2: 4


In [24]:
def answer_longest_palindrome(s: str) -> int:
    counts = {}
    for ch in s:
        counts[ch] = counts.get(ch, 0) + 1
    length, has_odd = 0, False
    for c in counts.values():
        length += c // 2 * 2
        if c % 2:
            has_odd = True
    return length + (1 if has_odd else 0)

def answer_can_permute(s: str) -> bool:
    odd = set()
    for ch in s:
        if ch in odd:
            odd.remove(ch)
        else:
            odd.add(ch)
    return len(odd) <= 1

def answer_longest_ignore_case(s: str) -> int:
    return answer_longest_palindrome(s.lower())

def answer_min_removals(s: str) -> int:
    return len(s) - answer_longest_palindrome(s)

assert answer_longest_palindrome("abccccdd") == 7 and answer_longest_palindrome("aabb") == 4
assert answer_can_permute("aab") is True and answer_can_permute("abc") is False
assert answer_longest_ignore_case("Aa") == 2 and answer_longest_ignore_case("abccccdd") == 7
assert answer_min_removals("abc") == 2 and answer_min_removals("aabb") == 0
print("Đáp án bài 3–6 PASS")

Đáp án bài 3–6 PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def longestPalindrome(self, s: str) -> int:
        counts = {}
        for ch in s:
            counts[ch] = counts.get(ch, 0) + 1
        length, has_odd = 0, False
        for c in counts.values():
            length += c // 2 * 2          # phần làm cặp
            if c % 2: has_odd = True
        return length + (1 if has_odd else 0)   # tối đa 1 ký tự ở giữa
```

**Checklist:**
- [ ] Phân biệt hoa/thường (không `lower()`).
- [ ] Số đếm lẻ vẫn đóng góp `c - 1`.
- [ ] Chỉ `+1` **một lần** và chỉ khi có số đếm lẻ.
- [ ] Trả **độ dài** (int), không phải chuỗi.
- [ ] Time `O(n)`, Space `O(1)` (tối đa 52 ký tự).